# prototype for iot api data pipe

In [ ]:
print('hi')

hello


## Try get data api

In [ ]:
from datetime import datetime, timedelta, timezone
import json
import httpx

BOX_ID = "6a1564a7253e80000738f37c"
BASE_URL = "https://api.opensensemap.org"


def get_last_10_min_data(box_id: str):
    # 1. Compute time window in UTC formatted as ISO 8601 (ending with 'Z')
    to_date = datetime.now(timezone.utc)
    from_date = to_date - timedelta(minutes=10)

    from_date_str = from_date.strftime("%Y-%m-%dT%H:%M:%SZ")
    to_date_str = to_date.strftime("%Y-%m-%dT%H:%M:%SZ")

    print(f"Fetching data from {from_date_str} to {to_date_str}")

    with httpx.Client(timeout=20.0) as client:
        # 2. Get box metadata to discover sensor IDs
        box_resp = client.get(f"{BASE_URL}/boxes/{box_id}?format=json")
        box_resp.raise_for_status()
        box_data = box_resp.json()

        sensors = box_data.get("sensors", [])
        box_measurements = {}


        box_measurements["device_id"] = box_id
        box_measurements["device_name"] = box_data.get("name")
        box_measurements["model"] = box_data.get("model")
        box_measurements["currentLocation"] = box_data.get("currentLocation")

        # 3. Fetch last 10 minutes for each sensor
        for sensor in sensors:
            sensor_id = sensor["_id"]
            title = sensor.get("title")
            unit = sensor.get("unit")
            sensor_type = sensor.get("sensorType")

            # Endpoint format:
            # https://api.opensensemap.org/boxes/:senseBoxId/data/:sensorId?from-date=fromDate&to-date=toDate&download=true&format=json
            data_url = f"{BASE_URL}/boxes/{box_id}/data/{sensor_id}"
            params = {
                "from-date": from_date_str,
                "to-date": to_date_str,
                "download": "true",
                "format": "json",
            }

            resp = client.get(data_url, params=params)
            resp.raise_for_status()
            measurements = resp.json()

            box_measurements[title] = {
                "metric_name": title,
                "ingest_time": to_date_str,
                "sensor_id": sensor_id,
                "sensor_type": sensor_type,
                "unit": unit,
                "data": measurements,
                "api_path": data_url
            }
            # print(f" - {title} ({sensor_id}): {len(measurements)} measurements found")


    return box_measurements


if __name__ == "__main__":
    result = get_last_10_min_data(BOX_ID)
    print(json.dumps(result, indent=2))


Fetching data from 2026-10-27T01:56:51Z to 2026-10-27T02:06:51Z
{
  "PM10": {
    "metric_name": "PM10",
    "ingest_time": "2026-10-27T02:06:51Z",
    "sensor_id": "6a1564a7253e80000738f37d",
    "sensor_type": "SDS 011",
    "unit": "\u00b5g/m\u00b3",
    "data": [],
    "api_path": "https://api.opensensemap.org/boxes/6a1564a7253e80000738f37c/data/6a1564a7253e80000738f37d"
  },
  "PM2.5": {
    "metric_name": "PM2.5",
    "ingest_time": "2026-10-27T02:06:51Z",
    "sensor_id": "6a1564a7253e80000738f37e",
    "sensor_type": "SDS 011",
    "unit": "\u00b5g/m\u00b3",
    "data": [],
    "api_path": "https://api.opensensemap.org/boxes/6a1564a7253e80000738f37c/data/6a1564a7253e80000738f37e"
  },
  "Temperatur": {
    "metric_name": "Temperatur",
    "ingest_time": "2026-10-27T02:06:51Z",
    "sensor_id": "6a1564a7253e80000738f37f",
    "sensor_type": "BME280",
    "unit": "\u00b0C",
    "data": [],
    "api_path": "https://api.opensensemap.org/boxes/6a1564a7253e80000738f37c/data/6a1564a72

In [5]:
from datetime import datetime, timedelta, timezone
import json
import httpx

BOX_ID = "6a1564a7253e80000738f37c"
BASE_URL = "https://api.opensensemap.org"


def get_last_10_min_data(box_id: str):
    # 1. Compute time window in UTC formatted as ISO 8601 (ending with 'Z')
    to_date = datetime.now(timezone.utc)
    from_date = to_date - timedelta(minutes=20)

    from_date_str = from_date.strftime("%Y-%m-%dT%H:%M:%SZ")
    to_date_str = to_date.strftime("%Y-%m-%dT%H:%M:%SZ")

    print(f"Fetching data from {from_date_str} to {to_date_str}")

    with httpx.Client(timeout=20.0) as client:
        # 2. Get box metadata to discover sensor IDs
        box_resp = client.get(f"{BASE_URL}/boxes/{box_id}?format=json")
        box_resp.raise_for_status()
        box_data = box_resp.json()

        sensors = box_data.get("sensors", [])
        box_measurements = {}

        # 3. Fetch last 10 minutes for each sensor
        for sensor in sensors:
            sensor_id = sensor["_id"]
            title = sensor.get("title")
            unit = sensor.get("unit")
            sensor_type = sensor.get("sensorType")

            # Endpoint format:
            # https://api.opensensemap.org/boxes/:senseBoxId/data/:sensorId?from-date=fromDate&to-date=toDate&download=true&format=json
            data_url = f"{BASE_URL}/boxes/{box_id}/data/{sensor_id}"
            params = {
                "from-date": from_date_str,
                "to-date": to_date_str,
                "download": "true",
                "format": "json",
            }

            resp = client.get(data_url, params=params)
            resp.raise_for_status()
            measurements = resp.json()

            box_measurements["measurements"] = {
                "metric_name": title,
                "ingest_time": to_date_str,
                "sensor_id": sensor_id,
                "sensor_type": sensor_type,
                "unit": unit,
                "data": measurements,
            }
            # print(f" - {title} ({sensor_id}): {len(measurements)} measurements found")

        return box_measurements


if __name__ == "__main__":
    result = get_last_10_min_data(BOX_ID)
    print(json.dumps(result, indent=2))


Fetching data from 2026-10-26T07:44:58Z to 2026-10-26T08:04:58Z
{
  "measurements": {
    "metric_name": "Luftdruck",
    "ingest_time": "2026-10-26T08:04:58Z",
    "sensor_id": "6a1564a7253e80000738f381",
    "sensor_type": "BME280",
    "unit": "Pa",
    "data": [
      {
        "location": [
          105.823413,
          21.070477,
          20
        ],
        "createdAt": "2026-10-26T08:04:29.511Z",
        "value": "101209.72"
      },
      {
        "location": [
          105.823413,
          21.070477,
          20
        ],
        "createdAt": "2026-10-26T08:01:47.827Z",
        "value": "101208.85"
      },
      {
        "location": [
          105.823413,
          21.070477,
          20
        ],
        "createdAt": "2026-10-26T07:59:00.404Z",
        "value": "101207.02"
      },
      {
        "location": [
          105.823413,
          21.070477,
          20
        ],
        "createdAt": "2026-10-26T07:56:12.598Z",
        "value": "101212.63"
      

In [ ]:
from datetime import datetime, timedelta, timezone
import json
import httpx

BOX_ID = "6a1564a7253e80000738f37c"
BASE_URL = "https://api.opensensemap.org"


def get_last_10_min_data(box_id: str):
    # 1. Compute time window in UTC formatted as ISO 8601 (ending with 'Z')
    to_date = datetime.now(timezone.utc)
    from_date = to_date - timedelta(minutes=10)

    from_date_str = from_date.strftime("%Y-%m-%dT%H:%M:%SZ")
    to_date_str = to_date.strftime("%Y-%m-%dT%H:%M:%SZ")

    print(f"Fetching data from {from_date_str} to {to_date_str}")

    with httpx.Client(timeout=20.0) as client:
        # 2. Get box metadata to discover sensor IDs
        box_resp = client.get(f"{BASE_URL}/boxes/{box_id}?format=json")
        box_resp.raise_for_status()
        box_data = box_resp.json()

        sensors = box_data.get("sensors", [])
        box_measurements = {}

        # 3. Fetch last 10 minutes for each sensor
        for sensor in sensors:
            sensor_id = sensor["_id"]
            title = sensor.get("title")
            unit = sensor.get("unit")

            # Endpoint format:
            # https://api.opensensemap.org/boxes/:senseBoxId/data/:sensorId?from-date=fromDate&to-date=toDate&download=true&format=json
            data_url = f"{BASE_URL}/boxes/{box_id}/data/{sensor_id}"
            params = {
                "from-date": from_date_str,
                "to-date": to_date_str,
                "download": "true",
                "format": "json",
            }

            resp = client.get(data_url, params=params)
            resp.raise_for_status()
            measurements = resp.json()
            
            box_measurements["measurements"] = {
                "measure_name": title,
                "sensor_id": sensor_id,
                "unit": unit,
                "count": len(measurements),
                "data": measurements,
            }

        return box_measurements


if __name__ == "__main__":
    result = get_last_10_min_data(BOX_ID)
    print(json.dumps(result, indent=2))


## ingest raw should not clean any

In [ ]:
from datetime import datetime, timedelta, timezone
import json
import httpx
from typing import Dict, Any, List, Optional

BOX_ID = "6a1564a7253e80000738f37c"
BASE_URL = "https://api.opensensemap.org"

# Standard sensor mapping to normalized metric names
SENSOR_KEY_MAP = {
    "PM10": "pm10_ug_m3",
    "PM2.5": "pm25_ug_m3",
    "Temperatur": "temperature_c",
    "rel. Luftfeuchte": "humidity_pct",
    "Luftdruck": "pressure_pa",
}

def get_last_10_min_data(box_id: str):
    # 1. Compute time window in UTC formatted as ISO 8601 (ending with 'Z')
    to_date = datetime.now(timezone.utc)
    from_date = to_date - timedelta(minutes=10)

    from_date_str = from_date.strftime("%Y-%m-%dT%H:%M:%SZ")
    to_date_str = to_date.strftime("%Y-%m-%dT%H:%M:%SZ")

    print(f"Fetching data from {from_date_str} to {to_date_str}")

    with httpx.Client(timeout=20.0) as client:
        # 2. Get box metadata to discover sensor IDs
        box_resp = client.get(f"{BASE_URL}/boxes/{box_id}?format=json")
        box_resp.raise_for_status()
        box_data = box_resp.json()

        sensors = box_data.get("sensors", [])
        box_measurements = {}
        
        device_name = box_data.get("name")

        # 3. Fetch last 10 minutes for each sensor
        for sensor in sensors:
            sensor_id = sensor["_id"]
            title = sensor.get("title")
            unit = sensor.get("unit")
            sensor_type = sensor.get("sensorType")


            # Endpoint format:
            # https://api.opensensemap.org/boxes/:senseBoxId/data/:sensorId?from-date=fromDate&to-date=toDate&download=true&format=json
            data_url = f"{BASE_URL}/boxes/{box_id}/data/{sensor_id}"
            params = {
                "from-date": from_date_str,
                "to-date": to_date_str,
                "download": "true",
                "format": "json",
            }

            resp = client.get(data_url, params=params)
            resp.raise_for_status()
            measurements = resp.json()

            box_measurements["measurements"] = {
                "metric_name": title,
                "sensor_id": sensor_id,
                "unit": unit,
                "count": len(measurements),

                "data": measurements,
            }
            wide_schema =                {
                    "device_id": BOX_ID,
                    "device_name": device_name,
                    "metric_name": title,
                    "sensor_id": sensor_id,
                    "sensor_type": sensor_type,
                    "unit": unit,
                    **measurements
                }

        return     {
                    "device_id": BOX_ID,
                    "device_name": device_name,
                    "metric_name": title,
                    "sensor_id": sensor_id,
                    "sensor_type": sensor_type,
                    "unit": unit,
                    **measurements
                }


if __name__ == "__main__":
    result = get_last_10_min_data(BOX_ID)
    print(json.dumps(result, indent=2))


Fetching data from 2026-10-26T07:20:04Z to 2026-10-26T07:30:04Z
 - PM10 (6a1564a7253e80000738f37d): 4 measurements found
 - PM2.5 (6a1564a7253e80000738f37e): 4 measurements found
 - Temperatur (6a1564a7253e80000738f37f): 4 measurements found
 - rel. Luftfeuchte (6a1564a7253e80000738f380): 4 measurements found
 - Luftdruck (6a1564a7253e80000738f381): 4 measurements found
{
  "PM10": {
    "sensor_id": "6a1564a7253e80000738f37d",
    "unit": "\u00b5g/m\u00b3",
    "count": 4,
    "data": [
      {
        "location": [
          105.823413,
          21.070477,
          20
        ],
        "createdAt": "2026-10-26T07:28:28.025Z",
        "value": "21.60"
      },
      {
        "location": [
          105.823413,
          21.070477,
          20
        ],
        "createdAt": "2026-10-26T07:25:46.085Z",
        "value": "19.40"
      },
      {
        "location": [
          105.823413,
          21.070477,
          20
        ],
        "createdAt": "2026-10-26T07:23:05.528Z",
 